<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [12]:
# Numerical arrays and vectorized operations used for triangle genomes and RMSE calculations.
import numpy as np
# Plotting library used to show final images and convergence curves inside the notebook.
import matplotlib.pyplot as plt
# DataFrame structure used to store and analyze results across multiple runs and configurations.
import pandas as pd
# Random utilities used by the genetic algorithm: initialization, selection, and mutation.
from random import randint, choices, sample, random
# OpenCV is used to load images, draw filled triangles, and save generated outputs.
import cv2
# ABC/abstractmethod are used to define a generic optimization solution template.
from abc import ABC, abstractmethod
# deepcopy prevents offspring/elites from accidentally sharing the same object in memory as parents.
from copy import deepcopy
# OS is used for file path manipulations and directory creation.
import os
# Time is used to measure the runtime of the algorithm and to timestamp output files.
import time

## <center> Simulated Annealing

In [13]:
class Solution(ABC):
    """
    Abstract base class defining the foundational blueprint for candidate solutions in optimization algorithms.

    This generic class establishes the required architecture for any evolutionary or single-point 
    optimization state. It mandates that all inheriting subclasses (such as `VermeerSolution`) 
    must implement their own specific data structure representation, random initialization logic, 
    and mathematical fitness evaluation.
    """

    def __init__(self, repr=None):
        """
        Initializes the generic solution instance and establishes its genetic representation.

        If a specific representation is not passed during instantiation, this method automatically 
        invokes the abstract `random_initial_representation()` method to generate a starting state.

        Internal Variables:
            self.repr (Any): The core data structure (genome/state) of the solution. The exact data 
                type (e.g., numpy.ndarray, list, bitstring) depends on the specific subclass implementation.

        Parameters:
            repr (Any | None): An optional pre-existing representation to clone or inject. If None, 
                a completely randomized state is generated.
        """
        # If no representation is provided, create a random one using the subclass method.
        if repr is None:
            repr = self.random_initial_representation()

        # Store the representation/genome of the solution.
        self.repr = repr

    def __repr__(self):
        """
        Provides a human-readable string output of the solution's internal representation.

        Returns:
            str: The string-cast output of the `self.repr` data structure, used primarily for debugging or logging.
        """
        # Defines how the solution is displayed when printed.
        return str(self.repr)

    @abstractmethod
    def fitness(self):
        # Every specific solution must define how quality/fitness is calculated.
        pass

    @abstractmethod
    def random_initial_representation(self):
        # Every specific solution must define how a random starting solution is created.
        pass

In [14]:
class VermeerSolution(Solution):
    """
    Represents a candidate painting composed of 100 semi-transparent colored triangles.

    This class handles the representation (state) of a single painting, the logic to randomly 
    initialize its 900 variables, the OpenCV rendering pipeline to convert the state into a 
    2D image matrix, and the mathematical evaluation functions required for optimization 
    algorithms like Simulated Annealing and Genetic Algorithms.
    """

    def __init__(self, target_image, repr=None):
        """
        Initializes a candidate solution and establishes the geometric constraints of the canvas.

        Internal Variables:
            self.target_image (numpy.ndarray): The reference image array used later for RMSE calculation.
            self.width (int): Hardcoded canvas width restriction (300 pixels).
            self.height (int): Hardcoded canvas height restriction (400 pixels).
            self.num_triangles (int): Hardcoded limit defining the state size (100 triangles).

        Parameters:
            target_image (numpy.ndarray): The target image matrix (BGR) the solution attempts to replicate.
            repr (numpy.ndarray | None): An optional pre-existing 100x9 state array. If None, 
                the superclass will trigger `random_initial_representation()` to build a new one.
        """
        # Store the original image
        self.target_image = target_image

        # Project canvas size
        self.width = 300
        self.height = 400
        self.num_triangles = 100

        super().__init__(repr=repr)

    def random_initial_representation(self):
        """
        Generates a completely randomized starting state of 100 clustered triangles.

        To prevent massive, screen-covering triangles during the initial state generation, 
        this method uses a bounding radius. It picks a random center point and generates 
        three vertices tightly grouped around that center.

        Internal Variables:
            random_repr (numpy.ndarray): A (100, 9) integer matrix initialized to zeros. 
                Each row holds [x1, y1, x2, y2, x3, y3, r, g, b].
            cx, cy (int): Randomly generated origin coordinates for a single triangle.
            radius (int): A pixel bound (40) restricting how far vertices can spawn from the center.
            x1, y1, x2, y2, x3, y3 (int): Coordinate pairs generated within the radius and clipped to the canvas.
            r, g, b (int): Randomly generated color channels between 0 and 255.

        Returns:
            numpy.ndarray: The populated 100x9 integer matrix representing the new initial state.
        """
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)
        for i in range(self.num_triangles):
            cx = randint(0, self.width)
            cy = randint(0, self.height)
            radius = 40
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)
            r, g, b = [randint(0, 255) for _ in range(3)]
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]
        return random_repr

    def render_canvas(self):
        """
        Parses the numerical state and paints it onto a 2D pixel matrix using OpenCV.

        The method iterates row-by-row through the state. The array sequence dictates the 
        drawing sequence (Z-index), meaning shapes at the end of the array will be drawn 
        on top of shapes at the beginning.

        Internal Variables:
            canvas (numpy.ndarray): A (400, 300, 3) matrix of unsigned 8-bit integers initialized to black.
            gene (numpy.ndarray): A 1D slice representing a single row (triangle) from `self.repr`.
            pts (numpy.ndarray): A 3x2 matrix of the triangle's integer coordinates, reshaped to (-1, 1, 2) 
                to comply with the specific contour formatting required by cv2.fillPoly.
            color (tuple): A tuple of three integers (R, G, B) extracted from the array.

        Returns:
            numpy.ndarray: The finalized 3D pixel matrix representing the visually rendered painting.
        """
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)
        for gene in self.repr:
            pts = np.array([[gene[0], gene[1]], [gene[2], gene[3]], [gene[4], gene[5]]], np.int32)
            pts = pts.reshape((-1, 1, 2))
            color = (int(gene[6]), int(gene[7]), int(gene[8]))
            cv2.fillPoly(canvas, [pts], color)
        return canvas

    def fitness(self):
        """
        Calculates the Root Mean Squared Error (RMSE) between the current state and the target image.

        To prevent memory overflow/underflow artifacts that occur when subtracting unsigned 8-bit 
        integers, the method mathematically upcasts the image matrices to 32-bit floats before 
        performing the pixel-wise subtraction.

        Internal Variables:
            generated_image (numpy.ndarray): The 3D pixel matrix output yielded by `self.render_canvas()`.
            target_float, gen_float (numpy.ndarray): The respective image matrices cast to `np.float32`.
            diff (numpy.ndarray): The raw pixel-by-pixel arithmetic difference between the two float matrices.
            sq_diff (numpy.ndarray): The `diff` matrix mathematically squared to eliminate negative values.
            mse (float): The Mean Squared Error, calculated as the statistical mean of `sq_diff`.
            rmse (float): The final Root Mean Squared Error, calculated as the square root of the MSE.

        Returns:
            float: The calculated visual error (RMSE). A value closer to 0.0 indicates higher visual accuracy.
        """
        generated_image = self.render_canvas()
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)

        # 1. Erro Visual (RMSE)
        diff = target_float - gen_float
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        return rmse

    def get_random_neighbor(self):
        """
        Generates a neighboring state specifically optimized for Single-Point Trajectory algorithms like Simulated Annealing.

        This method duplicates the current solution and applies a stochastic perturbation. 
        It uses a hybrid mutation strategy: primarily utilizing small uniform linear nudges for 
        local fine-tuning (exploitation), mixed with occasional complete triangle resets for 
        global search space exploration.

        Internal Variables:
            neighbor (VermeerSolution): A completely independent instantiated clone of the current solution.
            pm (float): The probability (0.03 or 3%) that any given triangle will be perturbed.
            mutation_type (float): A random float between 0.0 and 1.0 determining the perturbation scale.
                Values < 0.90 trigger a micro-nudge (±10 coords, ±15 colors) using `np.random.randint`.
                Values >= 0.90 trigger a full randomization of the triangle's 9 variables.

        Returns:
            VermeerSolution: The newly mutated neighboring state.
        """
        neighbor = VermeerSolution(
            target_image=self.target_image, 
            repr=self.repr.copy()
        )
        
        pm = 0.03 
        
        for i in range(neighbor.num_triangles):
            if random() < pm:
                mutation_type = random()
                
                if mutation_type < 0.90:
                    # 90% of mutations: Nudge
                    neighbor.repr[i][0:6] += np.random.randint(-10, 11, 6)  # Coordinates
                    neighbor.repr[i][6:9] += np.random.randint(-15, 16, 3)  # Colors
                    
                    # Secure bounds after mutation
                    neighbor.repr[i][0::2] = np.clip(neighbor.repr[i][0::2], 0, neighbor.width)
                    neighbor.repr[i][1::2] = np.clip(neighbor.repr[i][1::2], 0, neighbor.height)
                    neighbor.repr[i][6:9] = np.clip(neighbor.repr[i][6:9], 0, 255)
                    
                else:
                    # 10% of mutations: Random Reset
                    neighbor.repr[i] = [
                        randint(0, neighbor.width),
                        randint(0, neighbor.height),
                        randint(0, neighbor.width),
                        randint(0, neighbor.height),
                        randint(0, neighbor.width),
                        randint(0, neighbor.height),
                        randint(0, 255),  # Random red
                        randint(0, 255),  # Random green
                        randint(0, 255)   # Random blue
                    ]
                    
        return neighbor

In [15]:
def simulated_annealing(
    initial_solution: Solution,
    C: float,
    L: int,
    cooling_rate: float,
    max_iter: int = 10,
    verbose: bool = False,
):
    """
    Executes the Simulated Annealing optimization algorithm to minimize the RMSE of a candidate solution.

    This algorithm performs a stochastic random walk through the search space. It always accepts 
    state changes that improve the fitness (lower RMSE). To escape local optima, it occasionally 
    accepts worse states based on a dynamic Boltzmann probability distribution that decays as the 
    system "cools" over time. 

    Internal Variables:
        current_solution (Solution): The active state currently being explored in the random walk.
        current_fitness (float): The RMSE score of the active state.
        best_solution (Solution): A deep-copied vault storing the absolute lowest-RMSE state ever found.
        best_fitness (float): The lowest RMSE score ever recorded.
        rmse_history (list): A running log appending the `current_fitness` at every single evaluation step.
        total_evals (int): A counter tracking the absolute number of fitness evaluations performed.
        neighbor (Solution): The proposed next state, generated via `get_random_neighbor()`.
        delta (float): The mathematical difference in RMSE between the neighbor and the current state.
            A negative delta indicates an improvement (minimization).
        prob (float): The calculated acceptance probability (e^(-delta / C)) for a worse neighbor.

    Parameters:
        initial_solution (Solution): The starting state (usually a randomized `VermeerSolution`) to begin optimization.
        C (float): The initial Temperature. Higher values mean a higher initial probability of accepting worse solutions.
        L (int): The number of neighbor evaluations performed at each temperature plateau before C is modified.
        cooling_rate (float): The geometric decay factor (typically 0.80 to 0.99) multiplied against C after every L iterations.
        max_iter (int): The total number of times the temperature will be reduced (defaults to 10).
        verbose (bool): If True, prints a formatted progress log to the console every 50 temperature iterations.

    Returns:
        tuple: A data structure containing:
            - best_solution (Solution): The mathematically superior state found across all evaluations.
            - rmse_history (list): List of floats tracking the active state's RMSE progression over total evaluations.
    """
    # Start from the provided initial solution and calculate its fitness.
    current_solution = initial_solution
    current_fitness = current_solution.fitness()

    # Best global solution as SA can accept worse ones
    best_solution = initial_solution
    best_fitness = current_fitness

    C = C
    rmse_history = []  
    total_evals = 0

    for iteration in range(1, max_iter + 1):

        for _ in range(L):
            # Generate a neighbor with a slight mutation
            neighbor = current_solution.get_random_neighbor()
            neighbor_fitness = neighbor.fitness()
            total_evals += 1

            delta = neighbor_fitness - current_fitness   # less than 0, improvement

            if delta <= 0:
                # if the neighbor is better, accept it as current solution
                current_solution = neighbor
                current_fitness = neighbor_fitness
            else:
                # if the neighbor is worse, accept with probability of Boltzmann
                prob = np.exp(-delta / C) if C > 1e-10 else 0.0
                if random() < prob:
                    current_solution = neighbor
                    current_fitness = neighbor_fitness

            # Update global best
            if current_fitness < best_fitness:
                best_fitness = current_fitness
                # Save a copy of the best solution
                best_solution = VermeerSolution(
                    target_image=current_solution.target_image,
                    repr=current_solution.repr.copy(),
                )

            rmse_history.append(current_fitness)

        # Geometric cooling
        C *= cooling_rate

        if verbose and iteration % 50 == 0:
            print(
                f"  Iter {iteration:5d}/{max_iter} | "
                f"C={C:.4f} | "
                f"RMSE atual={current_fitness:.2f} | "
                f"Best={best_fitness:.2f}"
            )

    return best_solution, rmse_history


In [16]:
# Path to local copies of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"
rafa_path = r"Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
gui_path_2 = r"C:\Users\User\Documents\GitHub\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"

# Load the original image using OpenCV.
original_image = cv2.imread(gui_path_2)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {gui_path_2}")

In [20]:
import os
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import copy

# --- 1. CONFIGURAÇÃO DO LOTE ---
EXP_NAME = "Simulated_Annealing"
NOTES = "SA Optimization with 300.000 evaluations."
NUM_RUNS = 20

# Parâmetros Otimizados do SA
L_PARAM = 150         
MAX_ITER_PARAM = 2000 
C_INICIAL = 10.0      
COOLING_RATE_PARAM = 0.996

print(f"\n[{time.strftime('%H:%M:%S')}] Starting SA batch: {EXP_NAME} ({NUM_RUNS} Runs)")

variant_dir = os.path.join("data", "results", EXP_NAME)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

start_time = time.time()

# --- 2. CICLO SEQUENCIAL (Uma Run de cada vez) ---
for run in range(1, NUM_RUNS + 1):
    print(f"\nStarting SA Run {run}/{NUM_RUNS}...")
    
    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)
        
    # ⚠️ ALERTA DE JUSTIÇA ACADÉMICA (PONTO 4):
    # Neste momento, isto gera uma solução aleatória cega.
    # Quando fores comparar com o GA no final, substitui a linha abaixo por:
    # sa_initial = copy.deepcopy(melhor_quadro_inicial_do_ga)
    sa_initial = VermeerSolution(target_image=original_image)

    # Executar o SA (verbose=False para não encher o ecrã)
    best_sa_sol, sa_history_raw = simulated_annealing(
        initial_solution=sa_initial,
        C=C_INICIAL, 
        L=L_PARAM, 
        cooling_rate=COOLING_RATE_PARAM, 
        max_iter=MAX_ITER_PARAM, 
        verbose=True
    )
    
    # Processar o histórico para "Best-So-Far" (para o gráfico não andar aos saltos)
    best_so_far_sa = []
    current_best = sa_history_raw[0]
    for score in sa_history_raw:
        if score < current_best:
            current_best = score
        best_so_far_sa.append(current_best)
        
    final_rmse = best_so_far_sa[-1]
    print(f"SA Run {run} completed. Final RMSE: {final_rmse:.2f}")

    # Guardar Hiperparâmetros e Histórico
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"Simulated Annealing Hyperparameters | Run Number: {run}\n\n")
        if NOTES: file.write(f"Notes: {NOTES}\n\n")
        file.write(f"Initial Temperature (C): {C_INICIAL}\n")
        file.write(f"Cooling Rate: {COOLING_RATE_PARAM}\n")
        file.write(f"Iterations at each Temp (L): {L_PARAM}\n")
        file.write(f"Max Temperature Steps (max_iter): {MAX_ITER_PARAM}\n")
        file.write(f"Total Evaluations: {L_PARAM * MAX_ITER_PARAM}\n\n")
        
        # Guardamos de 100 em 100 iterações para o TXT não ficar gigantesco
        file.write(f"--- ITERATION HISTORY ---\n")
        step = (L_PARAM * MAX_ITER_PARAM) // 40 
        
        for i in range(0, len(best_so_far_sa), step):
            # Dividir por L_PARAM converte o número da avaliação para a "Iteração" equivalente (0, 50, 100...)
            equiv_iter = i // L_PARAM
            file.write(f"Iter {equiv_iter} | Fitness: {best_so_far_sa[i]:.2f}\n")
        file.write(f"\n--- FINAL RESULTS ---\nFinal RMSE Score: {final_rmse:.2f}\n")

    # Guardar Imagem Final
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sa_sol.render_canvas())
    
    # Guardar Gráfico de Convergência (Individual)
    plt.figure(figsize=(8, 4))
    plt.plot(best_so_far_sa, color='orange', linewidth=1.5)
    plt.title(f"SA Convergence - Run {run} (Final RMSE: {final_rmse:.2f})")
    plt.xlabel("Temperature Steps (max_iter)")
    plt.ylabel("RMSE (Best-so-Far)")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches='tight')
    plt.close()
    
    # Adicionar dados globais para os CSVs
    all_runs_history.append(best_so_far_sa)
    final_rmses.append(final_rmse)

history = [run_history[::L_PARAM] for run_history in all_runs_history]

# Garantir que ficam todos do mesmo tamanho antes de converter para numpy
min_len = min(len(r) for r in history)
aligned_history = [r[:min_len] for r in history]

print(f"\n[{time.strftime('%H:%M:%S')}] Finished SA batch: {EXP_NAME} ({NUM_RUNS} Runs)")
print(f"Best Mean RMSE: {np.mean(aligned_history, axis=0)[-1]:.2f}")

# Converter para DataFrame: Linhas = Runs, Colunas = Passos de Temperatura
raw_df = pd.DataFrame(aligned_history)
raw_df.index = [f"Run_{i+1}" for i in range(NUM_RUNS)]
raw_df.columns = [f"Gen_{i}" for i in range(min_len)]

# Filtrar apenas colunas múltiplas de 50
cols_50 = [col for col in raw_df.columns if int(col.split("_")[1]) % 50 == 0]
raw_df_50 = raw_df[cols_50]

raw_df_50.to_csv(os.path.join(variant_dir, "raw_fitness_data_50.csv"))

# Statistical Summary CSV
stats_df = pd.DataFrame({
    "Iteration": np.arange(min_len),
    "Mean_RMSE": np.mean(aligned_history, axis=0),
    "Std_Dev": np.std(aligned_history, axis=0),
    "Best_RMSE": np.min(aligned_history, axis=0)
})
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[12:52:06] Starting SA batch: Simulated_Annealing (20 Runs)

Starting SA Run 1/20...
  Iter    50/2000 | C=8.1840 | RMSE atual=100.75 | Best=81.35
  Iter   100/2000 | C=6.6978 | RMSE atual=93.21 | Best=81.35
  Iter   150/2000 | C=5.4815 | RMSE atual=87.84 | Best=81.08
  Iter   200/2000 | C=4.4861 | RMSE atual=94.52 | Best=80.03
  Iter   250/2000 | C=3.6714 | RMSE atual=91.00 | Best=80.03
  Iter   300/2000 | C=3.0047 | RMSE atual=90.78 | Best=79.57
  Iter   350/2000 | C=2.4591 | RMSE atual=80.89 | Best=75.10
  Iter   400/2000 | C=2.0125 | RMSE atual=83.50 | Best=75.10
  Iter   450/2000 | C=1.6470 | RMSE atual=91.33 | Best=74.91
  Iter   500/2000 | C=1.3479 | RMSE atual=72.68 | Best=67.70
  Iter   550/2000 | C=1.1032 | RMSE atual=76.52 | Best=67.70
  Iter   600/2000 | C=0.9028 | RMSE atual=75.56 | Best=67.70
  Iter   650/2000 | C=0.7389 | RMSE atual=71.08 | Best=66.38
  Iter   700/2000 | C=0.6047 | RMSE atual=71.23 | Best=63.95
  Iter   750/2000 | C=0.4949 | RMSE atual=63.02 | Best=58.4